# Turkish Legal QA Dataset Preparation

## Purpose

This notebook prepares and merges two Turkish legal QA datasets into a single, clean, normalized dataset suitable for fine-tuning and evaluation.

**Datasets:**
1. **Kaggle**: batuhankalem/turkish-law-dataset-for-llm-finetuning (CSV, user-provided)
2. **Hugging Face**: Renicames/turkish-law-chatbot (via API)

**Outputs:**
- `kaggle_normalized.csv` - Normalized Kaggle dataset
- `hf_normalized.csv` - Normalized Hugging Face dataset
- `merged_legal_qa.csv` - Combined normalized dataset
- `merged_legal_qa.jsonl` - Same data in JSONL format

**Processing Steps:**
1. Load and inspect both datasets
2. Normalize to common schema
3. Clean and standardize text fields
4. Remove duplicates
5. Perform quality checks
6. Save processed outputs

## 1. Imports and Dependencies

In [ ]:
import os
import json
import pandas as pd
from typing import Optional, Dict, List
from pathlib import Path
from datasets import load_dataset

## 2. Google Drive Setup and Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Configure data paths on Google Drive
DRIVE_PROJECT_ROOT = Path("/content/drive/My Drive/nlp-rag-project")
DATA_RAW = DRIVE_PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = DRIVE_PROJECT_ROOT / "data" / "processed"

# Ensure directories exist
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# File paths
KAGGLE_CSV_PATH = DATA_RAW / "turkish_law_dataset.csv"

# HuggingFace API settings
HF_DATASET = "Renicames/turkish-law-chatbot"
HF_CONFIG = "default"
HF_SPLIT = "train"
HF_BATCH_SIZE = 100
HF_MAX_ROWS = None  # None = fetch ALL rows
HF_API_URL = "https://datasets-server.huggingface.co/rows"
REQUEST_TIMEOUT = 30  # seconds
SLEEP_BETWEEN_REQUESTS = 1  # seconds

print(f"✓ Google Drive mounted")
print(f"Project root: {DRIVE_PROJECT_ROOT}")
print(f"Raw data path: {DATA_RAW}")
print(f"Processed data path: {DATA_PROCESSED}")
print(f"Kaggle CSV path: {KAGGLE_CSV_PATH}")

## 3. Load Kaggle Dataset from Google Drive

In [ ]:
# Load Kaggle dataset
kaggle_df = pd.read_csv(KAGGLE_CSV_PATH)
print(f"✓ Loaded Kaggle dataset: {kaggle_df.shape[0]} rows, {kaggle_df.shape[1]} columns")
print(f"\nColumns: {list(kaggle_df.columns)}")

## 4. Fetch Hugging Face Dataset via API

In [ ]:
# Load HF dataset using datasets library (avoids rate limiting)
print("Fetching Hugging Face dataset (full train split) ...")
hf_dataset = load_dataset("Renicames/turkish-law-chatbot", split="train")
hf_df = hf_dataset.to_pandas()

print(f"✓ Loaded HF dataset: {hf_df.shape[0]} rows, {hf_df.shape[1]} columns")
print(f"Columns: {list(hf_df.columns)}")
hf_df.head()

## 5. Inspect Dataset Schemas

In [ ]:
# Show Kaggle dataset sample and info
if kaggle_df is not None:
    print("="*80)
    print("KAGGLE DATASET")
    print("="*80)
    print(f"\nShape: {kaggle_df.shape}")
    print(f"\nData types:\n{kaggle_df.dtypes}")
    print(f"\nMissing values:\n{kaggle_df.isnull().sum()}")
    print(f"\nFirst row:\n{kaggle_df.iloc[0]}")
    print(f"\nSample:\n")
    print(kaggle_df.head(3))

In [ ]:
# Show HuggingFace dataset sample and info
print("="*80)
print("HUGGINGFACE DATASET")
print("="*80)
print(f"\nShape: {hf_df.shape}")
print(f"\nData types:\n{hf_df.dtypes}")
print(f"\nMissing values:\n{hf_df.isnull().sum()}")
print(f"\nFirst row:\n{hf_df.iloc[0]}")
print(f"\nSample:\n")
print(hf_df.head(3))

In [ ]:
def normalize_kaggle(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize Kaggle dataset to the common schema.
    Maps possible column variants to standard names.
    """
    result = pd.DataFrame()
    
    # Helper function to find and extract column (case-insensitive)
    def extract_column(df, variants):
        """
        Find and extract a column by checking multiple variants.
        variants: list of possible column names
        """
        cols_lower = {col.lower(): col for col in df.columns}
        for variant in variants:
            if variant.lower() in cols_lower:
                return df[cols_lower[variant.lower()]].copy()
        return pd.Series([None] * len(df))
    
    # Map columns with variants
    result["question"] = extract_column(df, ["soru", "question"])
    result["answer"] = extract_column(df, ["cevap", "answer"])
    result["source"] = extract_column(df, ["kaynak", "source"])
    result["category"] = extract_column(df, ["veri türü", "veri_türü", "category"])
    result["quality_score"] = extract_column(df, ["score", "quality_score"])
    
    # Add metadata
    result["dataset_name"] = "batuhankalem/turkish-law-dataset-for-llm-finetuning"
    result["split"] = "train"
    
    return result


def normalize_huggingface(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize Hugging Face dataset to the common schema.
    Maps 'Soru' -> 'question' and 'Cevap' -> 'answer'.
    """
    result = pd.DataFrame()
    
    # Map columns
    result["question"] = df.get("Soru", None)
    result["answer"] = df.get("Cevap", None)
    result["source"] = None
    result["category"] = None
    result["quality_score"] = None
    
    # Add metadata
    result["dataset_name"] = "Renicames/turkish-law-chatbot"
    result["split"] = "train"
    
    return result

## 6. Normalize Datasets to Common Schema

**Target Schema:**
- `question`: The legal question
- `answer`: The corresponding answer
- `source`: Document/reference source
- `category`: Legal category/domain
- `dataset_name`: Origin identifier
- `split`: Data split (train/test/val)
- `quality_score`: Quality rating (if available)

In [ ]:
# Normalize Kaggle
kaggle_normalized = normalize_kaggle(kaggle_df)
print(f"✓ Normalized Kaggle: {kaggle_normalized.shape[0]} rows")
print(f"  Columns: {list(kaggle_normalized.columns)}")

# Normalize HuggingFace
hf_normalized = normalize_huggingface(hf_df)
print(f"✓ Normalized HuggingFace: {hf_normalized.shape[0]} rows")
print(f"  Columns: {list(hf_normalized.columns)}")

## 7. Text Cleaning and Standardization

In [ ]:
def clean_text(text) -> Optional[str]:
    """
    Clean and standardize text fields.
    - Preserve Turkish characters
    - Strip leading/trailing whitespace
    - Collapse repeated spaces
    - Convert empty strings to None
    """
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return None
    
    if not isinstance(text, str):
        text = str(text)
    
    # Strip whitespace
    text = text.strip()
    
    # Collapse repeated spaces
    text = " ".join(text.split())
    
    # Return None for empty strings
    return text if text else None


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply text cleaning to a normalized dataframe.
    """
    df_clean = df.copy()
    
    # Clean text columns
    for col in ["question", "answer", "source", "category"]:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].apply(clean_text)
    
    return df_clean


def remove_incomplete_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove rows missing critical fields (question or answer).
    """
    initial_count = len(df)
    
    # Drop rows with missing question or answer
    df_clean = df.dropna(subset=["question", "answer"])
    
    removed = initial_count - len(df_clean)
    print(f"  Removed {removed} rows with missing question/answer")
    
    return df_clean


def remove_duplicate_pairs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove exact duplicate question-answer pairs (keep first occurrence).
    """
    initial_count = len(df)
    
    # Identify duplicates
    df_clean = df.drop_duplicates(subset=["question", "answer"], keep="first")
    
    removed = initial_count - len(df_clean)
    print(f"  Removed {removed} duplicate question-answer pairs")
    
    return df_clean


# Apply cleaning
if kaggle_normalized is not None:
    print(f"Cleaning Kaggle dataset (before: {kaggle_normalized.shape[0]} rows)")
    kaggle_normalized = clean_dataframe(kaggle_normalized)
    print(f"  After cleaning: {kaggle_normalized.shape[0]} rows")

print(f"Cleaning HuggingFace dataset (before: {hf_normalized.shape[0]} rows)")
hf_normalized = clean_dataframe(hf_normalized)
print(f"  After cleaning: {hf_normalized.shape[0]} rows")

## 8. Remove Missing Required Fields and Duplicates

In [ ]:
# Clean Kaggle
print(f"\nCleaning Kaggle (before: {kaggle_normalized.shape[0]})")
kaggle_normalized = remove_incomplete_rows(kaggle_normalized)
kaggle_normalized = remove_duplicate_pairs(kaggle_normalized)
print(f"  Final: {kaggle_normalized.shape[0]} rows")

# Clean HuggingFace
print(f"\nCleaning HuggingFace (before: {hf_normalized.shape[0]})")
hf_normalized = remove_incomplete_rows(hf_normalized)
hf_normalized = remove_duplicate_pairs(hf_normalized)
print(f"  Final: {hf_normalized.shape[0]} rows")

## 9. Merge Datasets

In [ ]:
# Merge both datasets
merged_df = pd.concat(
    [kaggle_normalized, hf_normalized],
    ignore_index=True
)
print(f"✓ Merged datasets:")
print(f"  Kaggle: {kaggle_normalized.shape[0]} rows")
print(f"  HuggingFace: {hf_normalized.shape[0]} rows")
print(f"  Total (before dedup): {merged_df.shape[0]} rows")

# Remove duplicates from merged dataset
merged_df = remove_duplicate_pairs(merged_df)
print(f"  Total (after dedup): {merged_df.shape[0]} rows")

## 10. Quality Checks and Statistics

In [ ]:
print("="*80)
print("DATA QUALITY REPORT")
print("="*80)

print(f"\nMerged Dataset Shape: {merged_df.shape}")
print(f"\nData Types:")
print(merged_df.dtypes)

print(f"\nMissing Values:")
print(merged_df.isnull().sum())

print(f"\nDataset Composition:")
print(merged_df["dataset_name"].value_counts())

print(f"\nQuality Score Statistics (if available):")
if merged_df["quality_score"].notna().any():
    print(merged_df["quality_score"].describe())
else:
    print("  No quality scores available")

print(f"\nCategory Distribution:")
print(merged_df["category"].value_counts().head(10))

print(f"\nText Field Lengths (characters):")
merged_df["question_len"] = merged_df["question"].str.len()
merged_df["answer_len"] = merged_df["answer"].str.len()
print(f"  Question - min: {merged_df['question_len'].min()}, max: {merged_df['question_len'].max()}, mean: {merged_df['question_len'].mean():.0f}")
print(f"  Answer   - min: {merged_df['answer_len'].min()}, max: {merged_df['answer_len'].max()}, mean: {merged_df['answer_len'].mean():.0f}")

# Remove temporary length columns
merged_df = merged_df.drop(columns=["question_len", "answer_len"])

In [ ]:
# Show random samples
print("\n" + "="*80)
print("RANDOM SAMPLES FROM MERGED DATASET")
print("="*80)

for idx, row in merged_df.sample(n=min(3, len(merged_df))).iterrows():
    print(f"\n--- Sample {idx} ({row['dataset_name']}) ---")
    print(f"Question: {row['question'][:100]}..." if len(str(row['question'])) > 100 else f"Question: {row['question']}")
    print(f"Answer: {row['answer'][:100]}..." if len(str(row['answer'])) > 100 else f"Answer: {row['answer']}")
    print(f"Category: {row['category']}")
    print(f"Source: {row['source']}")

## 11. Save Processed Outputs

In [ ]:
# Reorder columns for consistency
column_order = ["question", "answer", "source", "category", "dataset_name", "split", "quality_score"]
merged_df = merged_df[column_order]

print(f"Final DataFrame columns: {list(merged_df.columns)}")
print(f"Final shape: {merged_df.shape}")

In [ ]:
# Save individual normalized datasets
kaggle_output = DATA_PROCESSED / "kaggle_normalized.csv"
kaggle_normalized[column_order].to_csv(kaggle_output, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {kaggle_output}")

hf_output = DATA_PROCESSED / "hf_normalized.csv"
hf_normalized[column_order].to_csv(hf_output, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {hf_output}")

# Save merged dataset
merged_csv = DATA_PROCESSED / "merged_legal_qa.csv"
merged_df.to_csv(merged_csv, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {merged_csv}")

In [ ]:
# Save as JSONL
jsonl_output = DATA_PROCESSED / "merged_legal_qa.jsonl"

with open(jsonl_output, "w", encoding="utf-8") as f:
    for idx, row in merged_df.iterrows():
        json_record = row.to_dict()
        # Convert NaN to None for JSON serialization
        json_record = {
            k: (None if pd.isna(v) else v) for k, v in json_record.items()
        }
        f.write(json.dumps(json_record, ensure_ascii=False) + "\n")

print(f"✓ Saved: {jsonl_output}")
print(f"\nTotal records written: {len(merged_df)}")

## 12. Summary and Next Steps

In [ ]:
print("\n" + "="*80)
print("DATASET PREPARATION COMPLETE")
print("="*80)

print(f"\nProcessed Datasets Summary:")
print(f"  • Kaggle: {kaggle_normalized.shape[0]} samples")
print(f"  • HuggingFace: {hf_normalized.shape[0]} samples")
print(f"  • Merged (deduplicated): {merged_df.shape[0]} samples")

print(f"\nOutput Files:")
output_files = [
    ("kaggle_normalized.csv", True),
    ("hf_normalized.csv", True),
    ("merged_legal_qa.csv", True),
    ("merged_legal_qa.jsonl", True),
]
for fname, exists in output_files:
    status = "✓" if exists else "✗"
    print(f"  {status} {fname}")

print(f"\nLocation: {DATA_PROCESSED}")

print(f"\nNext Steps:")
print(f"  1. Review the merged dataset for quality and coverage")
print(f"  2. Build a retrieval corpus of Turkish legal documents")
print(f"  3. Create a retrieval evaluation benchmark")
print(f"  4. Implement ranking and retrieval components")
print(f"  5. Integrate with LLM for answer generation")
print(f"  6. Evaluate full RAG pipeline end-to-end")